# 🎯 Aula 2 — Filtragem Colaborativa por Usuário
### Módulo 06 · Engenharia de Software · Inteli 2026.1B

**Objetivo:** Implementar e comparar dois algoritmos de recomendação — **UserKNN** e **SVD**

---

**Entregas esperadas:**
1. Dataset sintético definido e carregado ✅
2. Exploração e análise dos dados ✅
3. Implementação do UserKNN ✅
4. Implementação do SVD ✅
5. Relatório comparativo (células Markdown ao final) ✅


## 📦 0. Instalação de Dependências

Execute a célula abaixo para instalar as bibliotecas necessárias.


In [ ]:
# Execute uma vez para instalar as dependências
!pip install scikit-surprise pandas numpy matplotlib seaborn scikit-learn


In [ ]:
# Imports principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt

# Configura plots
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

print('✅ Imports OK')


---
## 📋 Passo 1 — Defina seu Dataset

Antes de escrever qualquer código, responda às perguntas abaixo
**nesta célula Markdown** (edite diretamente):

| Campo | Sua resposta |
|---|---|
| **Nome do projeto** | |
| **Domínio** | ex: e-commerce, saúde, entretenimento, educação |
| **Quem são os "usuários"?** | ex: pacientes, clientes, alunos |
| **Quais são os "itens"?** | ex: produtos, artigos, músicas, remédios |
| **O que representa uma "avaliação"?** | ex: nota de 1–5, clique (0/1), tempo de uso |
| **Tamanho estimado do dataset** | ex: 50 usuários × 30 itens |
| **Por que recomendação faz sentido aqui?** | |


In [ ]:
# ─────────────────────────────────────────────────────
# TODO: Preencha a variável abaixo com a descrição
#       do seu dataset (apenas texto, para documentação)
# ─────────────────────────────────────────────────────

DATASET_DESCRICAO = {
    "projeto": "",          # Nome do projeto
    "dominio": "",           # Área de aplicação
    "entidade_usuario": "",  # Quem avalia
    "entidade_item": "",     # O que é avaliado
    "tipo_rating": "",       # Natureza da avaliação
    "escala": "",            # ex: 1-5, 0-1
    "n_usuarios_aprox": 0,
    "n_itens_aprox": 0,
}

print(DATASET_DESCRICAO)


---
## 🤖 Passo 2 — Geração de Dados Sintéticos com LLM

Use o prompt abaixo (ou adapte) em uma LLM de sua preferência
(ChatGPT, Claude, Gemini, etc.) para gerar um dataset sintético
no formato correto.

---

### 📝 Prompt sugerido

> Gere um dataset sintético de avaliações para um sistema de recomendação.
> O contexto é: **[descreva seu projeto]**.
>
> Requisitos:
> - **Usuários:** de 20 a 40 usuários com IDs como `u001`, `u002`, ...
> - **Itens:** de 15 a 30 itens com IDs como `i001`, `i002`, ... (dê nomes reais ao domínio)
> - **Avaliações:** escala de **[1–5 / 0–1 / 0–10]**, esparsidade de ~60%
>   (nem todo usuário avaliou todos os itens)
> - **Formato:** retorne um CSV com colunas `usuario_id, item_id, rating`
> - **Requisito extra:** injete preferências agrupadas (ex: 3 perfis de usuário
>   com gostos distintos) para que o algoritmo de recomendação faça sentido.

---

Cole o CSV gerado no arquivo `dataset_aula02.csv` na mesma pasta deste notebook.


In [ ]:
# ─────────────────────────────────────────────────────
# TODO: Carregue aqui o CSV gerado pelo LLM
# ─────────────────────────────────────────────────────

# Opção A — ler de arquivo CSV
# df = pd.read_csv('dataset_aula02.csv')

# Opção B — colar os dados diretamente (apenas se forem poucos)
# dados = """usuario_id,item_id,rating
# u001,i001,4
# ..."""
# from io import StringIO
# df = pd.read_csv(StringIO(dados))

df = None  # Substitua pelo carregamento acima

# ─────────────────────────────────────────────────────
# Verificação
assert df is not None, "❌ Carregue o dataset antes de continuar"
print(f"✅ Dataset carregado: {df.shape[0]} avaliações")
print(df.head())


---
## 🔍 Passo 3 — Exploração dos Dados (EDA)

Antes de modelar, entenda seus dados. Responda visualmente às perguntas abaixo.


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 3.1 — Informações gerais do dataset
# ─────────────────────────────────────────────────────

# Dica: use df.info(), df.describe(), df.nunique()



In [ ]:
# ─────────────────────────────────────────────────────
# TODO 3.2 — Distribuição dos ratings
# ─────────────────────────────────────────────────────

# Dica: use df['rating'].hist() ou sns.countplot(...)



In [ ]:
# ─────────────────────────────────────────────────────
# TODO 3.3 — Esparsidade da matriz
# Esparsidade = 1 - (n_avaliações / (n_usuarios * n_itens))
# ─────────────────────────────────────────────────────

n_usuarios = df['usuario_id'].nunique()
n_itens    = df['item_id'].nunique()
n_ratings  = len(df)

esparsidade = 1 - (n_ratings / (n_usuarios * n_itens))
print(f'Usuários únicos : {n_usuarios}')
print(f'Itens únicos    : {n_itens}')
print(f'Avaliações      : {n_ratings}')
print(f'Esparsidade     : {esparsidade:.1%}')


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 3.4 — Heatmap da matriz usuário-item (opcional)
# ─────────────────────────────────────────────────────

# Dica: use df.pivot_table(...) e sns.heatmap(...)



---
## 🧮 Passo 4 — Preparação: Matriz Usuário-Item e Split



In [ ]:
# ─────────────────────────────────────────────────────
# TODO 4.1 — Crie a matriz usuário-item (pivot table)
# Linhas = usuários, colunas = itens, valores = ratings
# Células sem avaliação = NaN
# ─────────────────────────────────────────────────────

# matriz = df.pivot_table(...)



In [ ]:
# ─────────────────────────────────────────────────────
# TODO 4.2 — Divida em treino e teste (80/20)
# Dica: mascare ~20% dos valores da matriz como NaN no
#       conjunto de treino e guarde-os como teste.
# ─────────────────────────────────────────────────────

# Sugestão: use train_test_split sobre o dataframe df
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)



---
## 👥 Passo 5 — UserKNN (k-Nearest Neighbors por Usuário)

### Como funciona:
1. Para cada par de usuários, calcule a **similaridade** (cosseno, Pearson, Jaccard…)
2. Para o usuário-alvo, encontre os **k vizinhos mais similares**
3. Preveja o rating de um item como a **média ponderada** dos ratings dos vizinhos

### Fórmula:
$$\hat{r}_{u,i} = \bar{r}_u + \frac{\sum_{v \in N(u)} sim(u,v) \cdot (r_{v,i} - \bar{r}_v)}{\sum_{v \in N(u)} |sim(u,v)|}$$


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 5.1 — Calcule a similaridade entre usuários
# Métricas sugeridas: cosine (sklearn) ou correlação de Pearson
# ─────────────────────────────────────────────────────

from sklearn.metrics.pairwise import cosine_similarity

# Preencha os NaN com 0 apenas para cálculo de similaridade
# matriz_filled = matriz.fillna(0)
# sim_matrix = cosine_similarity(matriz_filled)
# sim_df = pd.DataFrame(sim_matrix, index=matriz.index, columns=matriz.index)



In [ ]:
# ─────────────────────────────────────────────────────
# TODO 5.2 — Implemente a função de predição UserKNN
# ─────────────────────────────────────────────────────

def predict_userknn(usuario, item, matriz, sim_df, k=5):
    """
    Retorna o rating previsto para (usuario, item) usando UserKNN.
    
    Parâmetros
    ----------
    usuario : str   — ID do usuário-alvo
    item    : str   — ID do item a prever
    matriz  : DataFrame — matriz usuário-item (NaN onde não avaliou)
    sim_df  : DataFrame — matriz de similaridade entre usuários
    k       : int   — número de vizinhos
    
    Retorna
    -------
    float — rating previsto
    """
    # TODO: implemente aqui
    pass

# Teste rápido (ajuste os IDs conforme seu dataset)
# usuario_teste = df['usuario_id'].iloc[0]
# item_teste    = df['item_id'].iloc[0]
# pred = predict_userknn(usuario_teste, item_teste, matriz, sim_df, k=5)
# print(f'Predição para ({usuario_teste}, {item_teste}): {pred:.2f}')


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 5.3 — Avalie UserKNN no conjunto de teste (RMSE)
# ─────────────────────────────────────────────────────

def avaliar_userknn(test_df, matriz, sim_df, k=5):
    """Calcula RMSE do UserKNN sobre test_df."""
    y_real, y_pred = [], []
    for _, row in test_df.iterrows():
        pred = predict_userknn(row['usuario_id'], row['item_id'], matriz, sim_df, k)
        if pred is not None:
            y_real.append(row['rating'])
            y_pred.append(pred)
    rmse = sqrt(mean_squared_error(y_real, y_pred))
    return rmse, len(y_pred)

# rmse_knn, n_preditos = avaliar_userknn(test_df, matriz, sim_df, k=5)
# print(f'UserKNN — RMSE: {rmse_knn:.4f}  |  Predições: {n_preditos}')


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 5.4 — Gere as top-N recomendações para um usuário
# ─────────────────────────────────────────────────────

def recomendar_userknn(usuario, matriz, sim_df, k=5, top_n=5):
    """
    Retorna os top_n itens recomendados para o usuário,
    excluindo itens que ele já avaliou.
    """
    # TODO: implemente aqui
    pass

# recomendacoes = recomendar_userknn(usuario_teste, matriz, sim_df, k=5, top_n=5)
# print(f'Recomendações para {usuario_teste}:')
# print(recomendacoes)


---
## 🔢 Passo 6 — SVD (Fatoração de Matriz)

### Como funciona:
A matriz de ratings **R** é decomposta em três matrizes:

$$R \approx U \cdot \Sigma \cdot V^T$$

Onde **U** representa perfis de usuários e **V** representa perfis de itens
em um espaço de **fatores latentes**. A predição é o produto interno entre
os vetores latentes do usuário e do item.

### Opções de implementação:
- `scipy.linalg.svd` — SVD puro (baseline)
- `sklearn.decomposition.TruncatedSVD` — SVD truncado (mais eficiente)
- `surprise.SVD` — biblioteca especializada em recomendação


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 6.1 — Implemente SVD com sklearn (TruncatedSVD)
# ─────────────────────────────────────────────────────

from sklearn.decomposition import TruncatedSVD

def treinar_svd(matriz, n_fatores=10):
    """
    Treina um modelo SVD e retorna a matriz de ratings reconstruída.
    
    Parâmetros
    ----------
    matriz    : DataFrame — matriz usuário-item (NaN substituído por 0 ou média)
    n_fatores : int       — número de fatores latentes
    
    Retorna
    -------
    DataFrame — matriz reconstruída com predições para todos os pares
    """
    # TODO: implemente aqui
    # Dica: preencha NaN pela média do usuário antes do SVD
    pass

# matriz_reconstruida = treinar_svd(matriz, n_fatores=10)


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 6.2 — Avalie SVD no conjunto de teste (RMSE)
# ─────────────────────────────────────────────────────

def avaliar_svd(test_df, matriz_reconstruida):
    """Calcula RMSE do SVD sobre test_df."""
    y_real, y_pred = [], []
    for _, row in test_df.iterrows():
        try:
            pred = matriz_reconstruida.loc[row['usuario_id'], row['item_id']]
            y_real.append(row['rating'])
            y_pred.append(pred)
        except KeyError:
            pass
    rmse = sqrt(mean_squared_error(y_real, y_pred))
    return rmse, len(y_pred)

# rmse_svd, n_preditos_svd = avaliar_svd(test_df, matriz_reconstruida)
# print(f'SVD — RMSE: {rmse_svd:.4f}  |  Predições: {n_preditos_svd}')


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 6.3 — Gere as top-N recomendações com SVD
# ─────────────────────────────────────────────────────

def recomendar_svd(usuario, matriz, matriz_reconstruida, top_n=5):
    """
    Retorna os top_n itens recomendados via SVD,
    excluindo itens que o usuário já avaliou.
    """
    # TODO: implemente aqui
    pass

# rec_svd = recomendar_svd(usuario_teste, matriz, matriz_reconstruida, top_n=5)
# print(f'Recomendações SVD para {usuario_teste}:')
# print(rec_svd)


---
## 📊 Passo 7 — Avaliação e Comparação das Abordagens

Compare os dois modelos com as métricas abaixo.

| Métrica | Descrição |
|---|---|
| **RMSE** | Erro médio de predição (menor = melhor) |
| **Coverage** | % de pares usuário-item que o modelo consegue prever |
| **Top-N Overlap** | Proporção de itens em comum entre as top-N listas |


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 7.1 — Tabela comparativa de métricas
# ─────────────────────────────────────────────────────

# resultados = pd.DataFrame({
#     'Algoritmo': ['UserKNN', 'SVD'],
#     'RMSE': [rmse_knn, rmse_svd],
#     'N_preditos': [n_preditos, n_preditos_svd],
# })
# print(resultados.to_string(index=False))


In [ ]:
# ─────────────────────────────────────────────────────
# TODO 7.2 — Visualização comparativa
# ─────────────────────────────────────────────────────

# Dica: bar chart do RMSE, scatter das predições vs real, etc.



In [ ]:
# ─────────────────────────────────────────────────────
# TODO 7.3 — Compare as recomendações para um usuário
# Mostre lado a lado o que cada algoritmo recomenda
# ─────────────────────────────────────────────────────

# usuario_demo = df['usuario_id'].iloc[0]
# rec_knn = recomendar_userknn(usuario_demo, matriz, sim_df)
# rec_svd = recomendar_svd(usuario_demo, matriz, matriz_reconstruida)
# 
# comparacao = pd.DataFrame({'UserKNN': rec_knn, 'SVD': rec_svd})
# print(comparacao)


---
## 📝 Passo 8 — Relatório Comparativo

**Instruções:** Edite as células Markdown abaixo com suas análises.
Seja objetivo e fundamente suas conclusões nos resultados obtidos.


### 8.1 Descrição do Dataset

*[Descreva brevemente o dataset sintético gerado: domínio, número de usuários,
itens, avaliações, escala de rating, esparsidade.]*


### 8.2 Resultados Quantitativos

*[Preencha a tabela com os valores obtidos:]*

| Algoritmo | RMSE | Observações |
|---|---|---|
| UserKNN (k=5) | | |
| SVD (d=10) | | |

*[Qual algoritmo teve menor RMSE? A diferença foi expressiva?]*


### 8.3 Análise Qualitativa das Recomendações

*[Para o usuário de demonstração, as recomendações fazem sentido no contexto
do seu projeto? Quais itens os dois algoritmos recomendaram em comum?
Houve diferenças relevantes?]*


### 8.4 Vantagens e Limitações

**UserKNN:**
- ✅ *[cite pelo menos uma vantagem]*
- ⚠️ *[cite pelo menos uma limitação]*

**SVD:**
- ✅ *[cite pelo menos uma vantagem]*
- ⚠️ *[cite pelo menos uma limitação]*


### 8.5 Conclusão

*[Em qual cenário você escolheria UserKNN? E SVD?
O que mudaria se o dataset tivesse 10× mais usuários?
Como este sistema de recomendação poderia ser integrado ao seu projeto?]*
